In [3]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver


In [4]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=GROQ_API_KEY
)

In [5]:
class JokeState(TypedDict):

    topic:str
    joke:str
    explanation:str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'Write an explanation for the joke -{state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation':response}



In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke',generate_joke)
graph.add_node('generate_explanation',generate_explanation)

graph.add_edge(START,'generate_joke')
graph.add_edge('generate_joke','generate_explanation')
graph.add_edge('generate_explanation',END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable":{"thread_id":"1"}}

workflow.invoke({"topic":"pizza"},config = config1)

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words. "Crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. This is a literal characteristic of a pizza.\n\nHowever, "crusty" can also be used to describe someone\'s mood or personality, implying that they are irritable, grumpy, or unpleasant. For example, "He\'s been feeling a bit crusty all day, so let\'s give him some space."\n\nIn the joke, the punchline "it was feeling a little crusty" is a clever pun that connects the literal characteristic of a pizza\'s crust to the emotional state of being grumpy. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}

In [16]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. "Crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. This is a literal characteristic of a pizza.\n\nHowever, "crusty" can also be used to describe someone\'s mood or personality, implying that they are irritable, grumpy, or unpleasant. For example, "He\'s been feeling a bit crusty all day, so let\'s give him some space."\n\nIn the joke, the punchline "it was feeling a little crusty" is a clever pun that connects the literal characteristic of a pizza\'s crust to the emotional state of being grumpy. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id':

In [17]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood? \n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words. "Crusty" has a double meaning here. \n\nIn one sense, "crusty" refers to the outer layer of a pizza, which is typically crispy and crunchy. This is a literal characteristic of a pizza.\n\nHowever, "crusty" can also be used to describe someone\'s mood or personality, implying that they are irritable, grumpy, or unpleasant. For example, "He\'s been feeling a bit crusty all day, so let\'s give him some space."\n\nIn the joke, the punchline "it was feeling a little crusty" is a clever pun that connects the literal characteristic of a pizza\'s crust to the emotional state of being grumpy. The humor comes from the unexpected twist on the word\'s meaning, creating a clever and amusing connection between the setup and the punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id'

In [18]:
config2 = {"configurable":{"thread_id":"2"}}
workflow.invoke({"topic":"pasta"},config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.',
 'explanation': 'This joke is a play on words, using the physical properties of spaghetti to make a pun about relationships. \n\nSpaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. In the joke, the spaghetti refuses to get married because it\'s afraid of getting "tangled up" in a relationship. The phrase "tangled up" has a double meaning here:\n\n1. **Literal meaning**: Spaghetti can physically get tangled, which is a common experience when cooking or eating it.\n2. **Figurative meaning**: In relationships, "tangled up" can also mean becoming deeply involved or entwined emotionally, which can be complex and potentially messy.\n\nThe joke relies on this wordplay to create a humorous connection between the physical properties of spaghetti and the emotional complexities of a romantic relationship. It\'s a

In [19]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.', 'explanation': 'This joke is a play on words, using the physical properties of spaghetti to make a pun about relationships. \n\nSpaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. In the joke, the spaghetti refuses to get married because it\'s afraid of getting "tangled up" in a relationship. The phrase "tangled up" has a double meaning here:\n\n1. **Literal meaning**: Spaghetti can physically get tangled, which is a common experience when cooking or eating it.\n2. **Figurative meaning**: In relationships, "tangled up" can also mean becoming deeply involved or entwined emotionally, which can be complex and potentially messy.\n\nThe joke relies on this wordplay to create a humorous connection between the physical properties of spaghetti and the emotional complexities of a romantic re

In [20]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti refuse to get married?\n\nBecause it was afraid of getting tangled up in a relationship.', 'explanation': 'This joke is a play on words, using the physical properties of spaghetti to make a pun about relationships. \n\nSpaghetti is a type of long, thin, and flexible pasta that can easily become tangled or knotted. In the joke, the spaghetti refuses to get married because it\'s afraid of getting "tangled up" in a relationship. The phrase "tangled up" has a double meaning here:\n\n1. **Literal meaning**: Spaghetti can physically get tangled, which is a common experience when cooking or eating it.\n2. **Figurative meaning**: In relationships, "tangled up" can also mean becoming deeply involved or entwined emotionally, which can be complex and potentially messy.\n\nThe joke relies on this wordplay to create a humorous connection between the physical properties of spaghetti and the emotional complexities of a romantic r

Time Travel

In [22]:
workflow.get_state({"configurable":{"thread_id":"1","checkpoint_id":"1f127f2e-3cb4-6e66-8004-eaa061904dd1"}})

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'This joke is a play on words, using a common phrase "feeling a little crusty" which typically means being irritable or grumpy, but also referencing the fact that a pizza has a crust. \n\nIn this joke, the setup "Why was the pizza in a bad mood?" primes the listener to expect a reason for the pizza\'s bad mood. The punchline "Because it was feeling a little crusty" is a clever twist, as it uses the phrase "feeling a little crusty" in a literal sense (the pizza has a crust) while also referencing the common phrase for being irritable. This wordplay creates a humorous connection between the setup and the punchline, making it a clever and amusing joke.'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f127f2e-3cb4-6e66-8004-eaa061904dd1'}}, metadata={'source': 'loop', 'step': 4, 'parents': {}}, created_at='202

In [23]:
workflow.invoke(None,{"configurable":{"thread_id":"1","checkpoint_id":"1f127f2e-3cb4-6e66-8004-eaa061904dd1"}})

{'topic': 'pizza',
 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.',
 'explanation': 'The joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. Typically, when someone says they\'re feeling "crusty," it means they\'re feeling irritable, grumpy, or cranky. However, in this joke, the phrase takes on a double meaning because a pizza has a crust - the outer layer of the pizza that is crispy and golden brown.\n\nSo, the joke is saying that the pizza is in a bad mood, and the reason for this is that it\'s "feeling a little crusty." The word "crusty" here is used to make a humorous connection between the pizza\'s emotional state (being grumpy) and its physical composition (having a crust). The joke relies on this clever play on words to create a lighthearted and amusing punchline.'}

In [24]:
list(workflow.get_state_history(config=config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why was the pizza in a bad mood?\n\nBecause it was feeling a little crusty.', 'explanation': 'The joke is a play on words, using a common phrase "feeling a little crusty" to create a pun. Typically, when someone says they\'re feeling "crusty," it means they\'re feeling irritable, grumpy, or cranky. However, in this joke, the phrase takes on a double meaning because a pizza has a crust - the outer layer of the pizza that is crispy and golden brown.\n\nSo, the joke is saying that the pizza is in a bad mood, and the reason for this is that it\'s "feeling a little crusty." The word "crusty" here is used to make a humorous connection between the pizza\'s emotional state (being grumpy) and its physical composition (having a crust). The joke relies on this clever play on words to create a lighthearted and amusing punchline.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f127f47-bb82-667e-8006-441

Updating the state


In [33]:

workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f06cc6e-7232-6cb1-8000-f71609e6cec5", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f127f51-ea25-62f8-8000-fc7852017162'}}

In [34]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f127f51-ea25-62f8-8000-fc7852017162'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-25T02:48:35.962119+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='5394eeb3-53fe-7ee3-30b0-b3c477406b49', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f127f4c-e0cf-65f7-8000-83bd88d28e0d'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-25T02:46:20.765515+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id

In [36]:

list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f127f51-ea25-62f8-8000-fc7852017162'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-25T02:48:35.962119+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f06cc6e-7232-6cb1-8000-f71609e6cec5'}}, tasks=(PregelTask(id='5394eeb3-53fe-7ee3-30b0-b3c477406b49', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f127f4c-e0cf-65f7-8000-83bd88d28e0d'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-03-25T02:46:20.765515+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id